In [ ]:
# GNN Processing Core - End-to-End Demo

This notebook demonstrates a complete workflow using the GNN Processing Core for a real-world graph analysis task.

## Overview

We'll work through:
1. Loading and preprocessing a graph dataset
2. Feature engineering and extraction
3. Model selection and training
4. Performance monitoring
5. Making predictions
6. Visualizing results

Let's get started!


In [ ]:
# Import required libraries
import sys
sys.path.append('../..')  # Add project root to path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from datetime import datetime
import json

# Import GNN Processing Core components
from src.gnn.parser import GNNParser
from src.gnn.feature_extractor import NodeFeatureExtractor
from src.gnn.edge_processor import EdgeProcessor
from src.gnn.layers import GNNStack
from src.gnn.model_mapper import GraphToModelMapper
from src.gnn.batch_processor import GraphBatchProcessor, GraphData
from src.gnn.monitoring import get_monitor, monitor_performance
from src.gnn.metrics_collector import get_metrics_collector, GraphMetrics

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("GNN Processing Core loaded successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


In [ ]:
## 1. Create a Sample Social Network Dataset

Let's create a synthetic social network dataset to demonstrate the GNN processing capabilities.


In [ ]:
# Generate a synthetic social network
def create_social_network(num_users=100, avg_connections=5):
    """Create a synthetic social network with user features"""

    # Create a random graph
    G = nx.barabasi_albert_graph(num_users, avg_connections)

    # Generate user features
    users = []
    for i in range(num_users):
        user = {
            'id': i,
            'features': {
                'age': np.random.randint(18, 65),
                'activity_level': np.random.uniform(0, 1),
                'account_age_days': np.random.randint(1, 1000),
                'num_posts': np.random.randint(0, 500),
                'follower_count': G.degree(i),
                'verified': np.random.choice([0, 1], p=[0.9, 0.1]),
                'user_type': np.random.choice(['regular', 'influencer', 'business'],
                                            p=[0.7, 0.2, 0.1])
            },
            'label': np.random.choice([0, 1], p=[0.8, 0.2])  # Bot detection label
        }
        users.append(user)

    # Extract edges
    edges = list(G.edges())

    return users, edges, G

# Create the dataset
users, edges, G = create_social_network(num_users=200, avg_connections=5)

print(f"Created social network with {len(users)} users and {len(edges)} connections")
print(f"Average degree: {np.mean([d for n, d in G.degree()]):.2f}")
print(f"Network density: {nx.density(G):.4f}")

# Show sample user
print("\nSample user:")
print(json.dumps(users[0], indent=2))


In [ ]:
## 2. Feature Extraction

Now let's extract and normalize features from our social network data.


In [ ]:
# Configure feature extraction
feature_config = {
    'numerical_features': ['age', 'activity_level', 'account_age_days',
                          'num_posts', 'follower_count'],
    'categorical_features': ['verified', 'user_type'],
    'normalization': 'standard',
    'handle_missing': 'mean'
}

# Extract features
extractor = NodeFeatureExtractor()
extraction_result = extractor.extract_features(users, feature_config)

print(f"Extracted features shape: {extraction_result.features.shape}")
print(f"Feature names: {extraction_result.feature_names}")
print(f"\nFeature statistics:")
print(f"  Mean: {extraction_result.features.mean(0)}")
print(f"  Std:  {extraction_result.features.std(0)}")

# Extract labels
labels = torch.tensor([user['label'] for user in users], dtype=torch.long)
print(f"\nLabel distribution:")
print(f"  Class 0 (normal): {(labels == 0).sum().item()}")
print(f"  Class 1 (bot):    {(labels == 1).sum().item()}")


In [ ]:
## 3. Process Edges and Select Model Architecture

Let's process the graph edges and automatically select the best GNN architecture for our task.


In [ ]:
# Process edges
edge_processor = EdgeProcessor()
edge_index = edge_processor.to_edge_index(edges)

print(f"Edge index shape: {edge_index.shape}")
print(f"Number of edges: {edge_index.shape[1]}")

# Use automatic model selection
mapper = GraphToModelMapper()
graph_data = {
    'num_nodes': len(users),
    'num_edges': len(edges),
    'features': extraction_result.features,
    'edge_index': edge_index
}

mapping_result = mapper.map_graph_to_model(
    graph_data,
    task_type='node_classification'
)

print(f"\nModel Selection Results:")
print(f"  Recommended architecture: {mapping_result['architecture']}")
print(f"  Reason: {mapping_result['selection_reason']}")
print(f"  Suggested configuration:")
for key, value in mapping_result['config'].items():
    print(f"    {key}: {value}")


In [ ]:
## 4. Create and Train the GNN Model

Now let's create the GNN model and train it on our social network data.


In [ ]:
# Create the GNN model
model = GNNStack(
    input_dim=extraction_result.features.shape[1],
    output_dim=2,  # Binary classification
    architecture=mapping_result['architecture'],
    **mapping_result['config']
)

print(f"Model architecture: {model}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Setup training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
features = extraction_result.features.to(device)
edge_index = edge_index.to(device)
labels = labels.to(device)

# Split data
num_nodes = len(users)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

# 60% train, 20% val, 20% test
train_mask[:int(0.6 * num_nodes)] = True
val_mask[int(0.6 * num_nodes):int(0.8 * num_nodes)] = True
test_mask[int(0.8 * num_nodes):] = True

print(f"\nData split:")
print(f"  Train: {train_mask.sum().item()} nodes")
print(f"  Val:   {val_mask.sum().item()} nodes")
print(f"  Test:  {test_mask.sum().item()} nodes")
